# 9 — Concept Naming del MSAE

Notebook per la fase di **concept naming** del MSAE_RW addestrato su embedding CLIP ViT-L/14 di Places365.
Il modello è **selezionabile** nella cella di configurazione iniziale (§0): `EXPANSION_FACTOR=8` → latent_dim 6144,
oppure `EXPANSION_FACTOR=16` → latent_dim 12288.

**Cosa fa**: per ogni neurone del decoder MSAE assegna il termine del vocabolario **combinato
SUN Attributes + attribute_index.json** (~697 termini: 102 attributi di scena + 620 attributi
visivi, deduplicati) con massima cosine similarity rispetto alla direzione della feature
riportata nello spazio CLIP originale. Applica i filtri di validazione dell'Appendice A del
paper *Interpreting CLIP with Hierarchical Sparse Autoencoders* (Zaigrajew et al., ICML 2025).

Rispetto al LAION unigram (~37k termini generici), questo vocabolario è interamente composto
da attributi rilevanti per scene e oggetti: ci aspettiamo top1_sim più alte e un maggior
numero di neuroni che superano la soglia paper (0.42).

## 0. Selezione del modello MSAE

Cella di configurazione: scegli **quale checkpoint MSAE** usare impostando `EXPANSION_FACTOR`.
Tutti i path a valle (nome del checkpoint) derivano da `LATENT_DIM = 768 × EXPANSION_FACTOR`.

- `EXPANSION_FACTOR = 8`  → `LATENT_DIM = 6144`  (modello del paper)
- `EXPANSION_FACTOR = 16` → `LATENT_DIM = 12288` (modello a capacità latente raddoppiata)

In [ ]:
# === SELEZIONE DEL MODELLO MSAE ===
# Imposta EXPANSION_FACTOR per scegliere quale checkpoint usare:
#   EXPANSION_FACTOR = 8   ->  LATENT_DIM = 6144   (modello del paper)
#   EXPANSION_FACTOR = 16  ->  LATENT_DIM = 12288  (modello a capacità latente raddoppiata)
EXPANSION_FACTOR = 16   # <-- cambia qui: 8 oppure 16

assert EXPANSION_FACTOR in (8, 16), f"EXPANSION_FACTOR deve essere 8 o 16, non {EXPANSION_FACTOR}"
LATENT_DIM = 768 * EXPANSION_FACTOR   # 768*8 = 6144 ; 768*16 = 12288
print(f"Modello MSAE selezionato: expansion_factor={EXPANSION_FACTOR}  ->  latent_dim={LATENT_DIM}")

## 1. Setup Colab — Drive, repo, dipendenze

Monta Google Drive, clona la repo MSAE in `/content/MSAE` , installa `openai-clip` se mancante e aggiunge la repo a `sys.path`.

In [ ]:
import os
import sys
import subprocess
import pathlib

from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = pathlib.Path('/content/MSAE')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WolodjaZ/MSAE.git', str(REPO_DIR)], check=True)
else:
    print(f"Repo gia' presente: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# openai-clip non e' preinstallato su Colab: serve solo se SAE/utils lo importano indirettamente.
try:
    import clip  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'openai-clip'], check=True)
    import clip  # noqa: F401

print(f'cwd = {os.getcwd()}')
print(f'sys.path[0] = {sys.path[0]}')

## 2. Configurazione path

Tutte le path del notebook sono definite qui come stringhe configurabili. Nessun path hardcoded altrove.

In [ ]:
import glob
import re

DRIVE_ROOT = '/content/drive/MyDrive/tesi'

# Checkpoint MSAE_RW selezionato in §0 (LATENT_DIM = 6144 o 12288) — naming convention della repo.
SAE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/models/{LATENT_DIM}_768_TopKReLU_64_RW_False_False_0.0_places365_ViT-L~14_train_image_20475_768.pth'

# Vocabolario combinato prodotto dal notebook 8 (SUN + attribute_index).
# N è dedotto dal nome file via glob: cambiando vocabolario non va aggiornato a mano.
VOCAB_NAME = "sun_attrindex"
emb_matches = sorted(glob.glob(f'{DRIVE_ROOT}/{VOCAB_NAME}_ViT-L~14_text_*_768.npy'))
assert len(emb_matches) == 1, (
    f'Atteso esattamente 1 file embedding per "{VOCAB_NAME}", trovati: {emb_matches}. '
    'Verifica di aver eseguito il notebook 8 e che non ci siano file duplicati su Drive.'
)
VOCAB_EMB_PATH = emb_matches[0]

# Estrae N dal nome (..._text_{N}_768.npy) per costruire il path dei termini allineati.
m = re.search(r'_text_(\d+)_768\.npy$', os.path.basename(VOCAB_EMB_PATH))
N_VOCAB = int(m.group(1))
VOCAB_TERMS_PATH = f'{DRIVE_ROOT}/{VOCAB_NAME}_ViT-L~14_text_{N_VOCAB}.txt'

# Directory di output per matrice di similarita', concept map e grafici.
OUTPUT_DIR = f'{DRIVE_ROOT}'
CHARTS_DIR = f'{OUTPUT_DIR}/charts'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

for p in [SAE_CHECKPOINT_PATH, VOCAB_EMB_PATH, VOCAB_TERMS_PATH]:
    assert os.path.exists(p), f'File mancante: {p}'
print(f'Modello (expansion={EXPANSION_FACTOR}, latent={LATENT_DIM}): {os.path.basename(SAE_CHECKPOINT_PATH)}')
print(f'Vocabolario: {VOCAB_NAME}  (N={N_VOCAB} termini)')
print('Tutti i file richiesti sono presenti.')

## 3. Caricamento modello SAE

Usa la classe `SAE` di `sae.py`: il filename del checkpoint codifica gli iperparametri (`n_latents=LATENT_DIM`, cioè 6144 o 12288 a seconda di `EXPANSION_FACTOR`; `n_inputs=768`, `activation=TopKReLU_64`, ecc.) e `load_model` li parsa automaticamente.

**Nota sul device**: `compute_similarities` (sae_naming.py:131-135) consuma direttamente i tensori prodotti dal `DataLoader` (CPU memmap) **senza** spostarli su GPU. Per evitare il mismatch di device con `decoded_search_space`, il modello viene mantenuto su CPU , questa e' la stessa scelta del flusso `main()` della repo. 

In [ ]:
import torch
from sae import SAE
from utils import set_seed

set_seed(42)

#Carico il modello e forzo a false weights only in quanto all'interno del file ci sono altre informazioni importanti. 
#(numpy scalars come mean_center, scaling_factor)
_orig_torch_load = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_compat

try:
    model = SAE(SAE_CHECKPOINT_PATH)
finally:
    torch.load = _orig_torch_load

 #sposta tutto il modello su CPU,Di base verrebbe processato metà su GPU e metà su CPU. Questo causa problemi. 
model = model.to('cpu')
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# NB: il modello resta su CPU per compatibilita' con compute_similarities della repo

print(f'device (informativo)         : {device}')
print(f'model.input_dim              : {model.input_dim}')
print(f'model.latent_dim             : {model.latent_dim}')
print(f'model.mean.shape             : {tuple(model.mean.shape)}')
print(f'model.mean.device            : {model.mean.device}')
print(f'model.scaling_factor         : {float(model.scaling_factor):.6f}')
print(f'model.scaling_factor.device  : {model.scaling_factor.device}')
print(f'model.model.decoder.device   : {model.model.decoder.device}')

## 4. Caricamento vocabolario combinato

Carica gli embedding con `SAEDataset(..., mean_center=False, target_norm=0.0)`.
La matrice di similarità sarà `[N_VOCAB, LATENT_DIM]` (con `N_VOCAB ≈ 697` per SUN + attribute_index,
e `LATENT_DIM` = 6144 o 12288 secondo il modello selezionato):
ogni neurone viene confrontato con tutti i termini — di scena e visivi — del vocabolario combinato.

In [ ]:
from utils import SAEDataset

dataset = SAEDataset(VOCAB_EMB_PATH, mean_center=False, target_norm=0.0)

with open(VOCAB_TERMS_PATH, 'r', encoding='utf-8') as f:
    terms = [line.strip() for line in f if line.strip()]

print(f'len(dataset) = {len(dataset)}')
print(f'len(terms)   = {len(terms)}')
assert len(dataset) == len(terms), f'Mismatch: {len(dataset)} embedding vs {len(terms)} termini'

first = dataset[0]
print(f'dataset[0].shape = {tuple(first.shape)}  dtype={first.dtype}')
print(f'Primi 5 termini: {terms[:5]}')

## 5. Nota metodologica — allineamento degli spazi

La repo `WolodjaZ/MSAE` realizza il concept naming con una strategia di allineamento degli spazi che **si discosta deliberatamente dall'Appendice A del paper**. Documento qui le tre scelte chiave, citando `sae_naming.py:119-124`.(invece di centrare e normalizzare gli embedding del vocabolario, "decentro e denormalizzo" quelli delle immagini per riportarle sullo spazio del clip e fare il confronto li)

### 5.1 Decoder portato nello spazio CLIP, vocabolario grezzo
Invece di portare gli embedding del vocabolario nello spazio interno del SAE (centrato e scalato), sono **le colonne del decoder** ad essere riportate nello spazio CLIP originale tramite `postprocess()`:

```python
# sae_naming.py:120-121
decoded_search_space = model.model.decoder.data + model.model.pre_bias
decoded_search_space = model.postprocess(decoded_search_space)
# postprocess: x / scaling_factor + mean
```

Coerentemente, il vocabolario e' caricato con `SAEDataset(vocab, mean_center=False, target_norm=0.0)` (`sae_naming.py:166`): gli embedding restano nello spazio CLIP originale. Un commento esplicito degli autori (`sae_naming.py:132-133`) motiva la scelta:

> *In the paper we used `input_processed, _ = model.model.preprocess(batch_data.to(device))`. However due to not perfect reconstruction of the model, we simply use the CLIP text [embeddings raw].*

Quindi: la repo **non** applica il preprocessing prescritto in Appendice A del paper, perche' la ricostruzione non e' perfetta e introdurrebbe rumore nel matching. Compensa portando il decoder nello spazio originale via `postprocess()`.

### 5.2 `patch_diff` — rimozione della componente di background

```python
# sae_naming.py:122-124
zero_space = model.decode(torch.zeros(1, model.latent_dim, ...))
decoded_search_space -= zero_space
```

Sottrae la ricostruzione del vettore latente nullo (che corrisponde a `pre_bias` riportato nello spazio CLIP). Rimuove cosi' la componente di ricostruzione **comune a tutti i neuroni** e rende le direzioni residue piu' specifiche della singola feature. Il default nella repo e' `patch_diff=True`; il paper originale non lo usava ma gli autori della repo lo trovano *piu' interpretabile* (`sae_naming.py:103-104`).

### 5.3 `pre_bias` mantenuto (Appendice A del paper)

Il decoder "semantico" e' `decoder + pre_bias`, **non** la sola matrice `decoder`. L'Appendice A del paper mostra empiricamente che sommare `pre_bias` prima della de-normalizzazione porta le similarity dei top match da ~0.1 a >0.42 e migliora la qualita' delle assegnazioni. Sebbene la repo si discosti dal paper sul preprocessing (5.1), su questo punto e' allineata: somma `pre_bias` prima di `postprocess`.



## 6. Calcolo matrice di similarita'

Importa `compute_similarities` e `cosine_similarity_matrix` direttamente da `sae_naming.py`. Esegue il forward sul vocabolario con `patch_diff=True`. Su Colab T4 riduco `batch_size=512` e `num_workers=2` (default repo: 4096 e 12) per stare nei limiti di VRAM e CPU.

Output: matrice float32 di shape `[n_vocab, n_neurons]` = `[~697, LATENT_DIM]`, salvata come `similarity_matrix.npy` in `OUTPUT_DIR`.

In [ ]:
import numpy as np
from sae_naming import compute_similarities, cosine_similarity_matrix  # noqa: F401

similarity_matrix = compute_similarities(
    model=model,
    dataset=dataset,
    batch_size=512,
    patch_diff=True,
    num_workers=2,
)

# Nome con LATENT_DIM: distingue i due modelli (6144 vs 12288) e ne evita la collisione.
sim_out_path = os.path.join(OUTPUT_DIR, f'similarity_matrix_{LATENT_DIM}.npy')
np.save(sim_out_path, similarity_matrix)

print(f'similarity_matrix.shape = {similarity_matrix.shape}  dtype={similarity_matrix.dtype}')
print(f'salvata in: {sim_out_path}')

## 6bis. Diagnostica 

verifica:
1. Distribuzione globale di top1_sim (max, percentili) — se max < 0.42 il decoder non ha direzioni allineate al vocabolario.
2. Parametri di postprocess (`mean`, `scaling_factor`) — se sono identita', il match e' tra vocab CLIP grezzo e decoder grezzo.
3. Norme delle righe del decoder — se hanno alta varianza o sono molto piccole, il training non ha convergato.
4. Norme degli embedding del vocabolario — devono essere ~1.0 (L2-normalizzati).

In [ ]:
import numpy as np

# --- 1. Distribuzione globale di similarity_matrix ---
sm = similarity_matrix  # [n_vocab, n_neurons]
print('=== Distribuzione globale similarity_matrix ===')
print(f'  shape          : {sm.shape}')
print(f'  min / max      : {sm.min():.4f} / {sm.max():.4f}')
print(f'  mean / std     : {sm.mean():.4f} / {sm.std():.4f}')
print(f'  pct 50/90/99   : {np.percentile(sm, [50, 90, 99])}')

# --- 2. Top1_sim per neurone (preview prima dei filtri) ---
sim_by_neuron_diag = sm.T  # [n_neurons, n_vocab]
top1_diag = sim_by_neuron_diag.max(axis=1)
print(f'\n=== top1_sim per neurone ({LATENT_DIM} valori) ===')
print(f'  min / max      : {top1_diag.min():.4f} / {top1_diag.max():.4f}')
print(f'  mean / std     : {top1_diag.mean():.4f} / {top1_diag.std():.4f}')
print(f'  pct 50/90/99   : {np.percentile(top1_diag, [50, 90, 99])}')
print(f'  # neuroni con top1 > 0.30 : {(top1_diag > 0.30).sum()}')
print(f'  # neuroni con top1 > 0.35 : {(top1_diag > 0.35).sum()}')
print(f'  # neuroni con top1 > 0.40 : {(top1_diag > 0.40).sum()}')
print(f'  # neuroni con top1 > 0.42 : {(top1_diag > 0.42).sum()}  <-- soglia paper')

# --- 3. Parametri postprocess ---
mean_t = model.mean.detach().cpu().numpy()
sf_t = float(model.scaling_factor)
is_identity = (np.abs(mean_t).max() < 1e-6) and (abs(sf_t - 1.0) < 1e-6)
postproc_kind = "identita'" if is_identity else "trasformazione attiva"
print('\n=== Parametri postprocess del SAE ===')
print(f'  mean.shape       : {mean_t.shape}')
print(f'  mean abs max     : {np.abs(mean_t).max():.6f}  (se ~0 -> nessun centering applicato)')
print(f'  scaling_factor   : {sf_t:.6f}  (se ~1 -> nessuno scaling)')
print(f'  postprocess(x) = x / {sf_t:.4f} + mean  ->  {postproc_kind}')

# --- 4. Norme delle righe del decoder ---
dec = model.model.decoder.data.detach().cpu().numpy()  # [n_latents, n_inputs]
dec_norms = np.linalg.norm(dec, axis=1)
print('\n=== Norme righe decoder ===')
print(f'  shape decoder       : {dec.shape}')
print(f'  norma min/max       : {dec_norms.min():.4f} / {dec_norms.max():.4f}')
print(f'  norma mean/std      : {dec_norms.mean():.4f} / {dec_norms.std():.4f}')
print(f'  # righe ~zero (<1e-4): {(dec_norms < 1e-4).sum()}  (dead neurons probabili)')

# --- 5. Norme degli embedding del vocabolario ---
vocab_arr = np.array([dataset[i].numpy() for i in range(0, len(dataset), max(1, len(dataset)//500))])
vocab_norms = np.linalg.norm(vocab_arr, axis=1)
print('\n=== Norme vocab (campione di ~500) ===')
print(f'  norma min/max  : {vocab_norms.min():.4f} / {vocab_norms.max():.4f}')
print(f'  norma mean/std : {vocab_norms.mean():.4f} / {vocab_norms.std():.4f}')
print(f'  (atteso: ~1.0 se L2-normalizzati)')

# --- 6. Interpretazione automatica ---
print('\n=== Diagnosi ===')
if top1_diag.max() < 0.42:
    msg = f'  top1_sim max = {top1_diag.max():.4f} < 0.42 -> NESSUN neurone supera la soglia paper.'
    print(msg)
    if (dec_norms < 1e-4).sum() > 100:
        print("  >> Molti dead neurons: il training MSAE non e' convergato sul subset attuale.")
    elif top1_diag.max() < 0.25:
        print('  >> top1 molto basso: il decoder non ha appreso direzioni semantiche del vocabolario.')
        print(f'     Causa probabile: dataset di training troppo piccolo (20k sample / {LATENT_DIM} latenti).')
    else:
        print("  >> top1 vicino alla soglia: training sotto-addestrato, prova ad abbassare")
        print("     temporaneamente la soglia (es. THR_SIM=0.30) per ottenere assegnazioni indicative,")
        print("     oppure riallena su un subset piu' grande.")
else:
    print(f'  top1_sim max = {top1_diag.max():.4f} >= 0.42 -> alcuni neuroni dovrebbero passare.')
    print('  Se n_valid=0 il problema e\' il filtro ratio>2.0: controlla la distribuzione di ratio.')

## 7. Trasposta e filtri di validazione

`compute_similarities` restituisce shape `[n_vocab, n_neurons]` = `[N_VOCAB, LATENT_DIM]`.
Per ragionare *per neurone* trasponiamo a `[n_neurons, n_vocab]`.

### 7.1 Filtri "paper" (Appendice A) — riferimento rigoroso
1. **top1_sim > 0.42** (soglia assoluta)
2. **top1_sim / top2_sim > 2.0** (margine di separazione)

Con un vocabolario interamente fatto di attributi pertinenti (vs ~37k termini generici LAION)
ci aspettiamo top1_sim più alte: è probabile che la soglia paper (0.42) sia raggiungibile
anche col checkpoint attuale.

### 7.2 Filtri "adattivi" (fallback se top1_sim < 0.42)

- **`THR_SIM_ADAPT = top1_sim.mean() + 3 * top1_sim.std()`**
- **`THR_RATIO_ADAPT = 1.10`**

Mantenuti come fallback per robustezza, ma con questo vocabolario ci aspettiamo
che la maggior parte dell'analisi possa usare direttamente i filtri paper.

In [ ]:
# Trasposta: [n_vocab, n_neurons] -> [n_neurons, n_vocab]
sim_by_neuron = similarity_matrix.T  # [LATENT_DIM, ~697]
n_neurons, n_vocab = sim_by_neuron.shape
print(f'sim_by_neuron.shape = {sim_by_neuron.shape}')

# Per ogni neurone trovo i primi due indici di vocabolario per similarita' decrescente.
top2_idx = np.argpartition(-sim_by_neuron, kth=1, axis=1)[:, :2]
top2_vals = np.take_along_axis(sim_by_neuron, top2_idx, axis=1)
order = np.argsort(-top2_vals, axis=1)
top2_idx_sorted = np.take_along_axis(top2_idx, order, axis=1)
top2_vals_sorted = np.take_along_axis(top2_vals, order, axis=1)

top1_idx = top2_idx_sorted[:, 0]
top1_sim = top2_vals_sorted[:, 0]
top2_sim = top2_vals_sorted[:, 1]

eps = 1e-12
ratio = top1_sim / np.maximum(top2_sim, eps)

# --- Filtri "paper" (Appendice A) ---
THR_SIM_PAPER = 0.42
THR_RATIO_PAPER = 2.0
valid_paper = (top1_sim > THR_SIM_PAPER) & (ratio > THR_RATIO_PAPER)

# --- Filtri "adattivi" (debug modello sotto-addestrato) ---
THR_SIM_ADAPT = float(top1_sim.mean() + 3.0 * top1_sim.std())
THR_RATIO_ADAPT = 1.10
valid_adaptive = (top1_sim > THR_SIM_ADAPT) & (ratio > THR_RATIO_ADAPT)

n_valid_paper = int(valid_paper.sum())
n_valid_adapt = int(valid_adaptive.sum())

print('\n=== Statistiche distribuzioni ===')
print(f'  top1_sim  mean={top1_sim.mean():.4f}  std={top1_sim.std():.4f}  max={top1_sim.max():.4f}')
print(f'  ratio     mean={ratio.mean():.4f}     std={ratio.std():.4f}     max={ratio.max():.4f}')

print('\n=== Filtri paper ===')
print(f'  THR_SIM_PAPER  = {THR_SIM_PAPER}')
print(f'  THR_RATIO_PAPER = {THR_RATIO_PAPER}')
print(f'  Neuroni validi (paper)   : {n_valid_paper}  ({100.0 * n_valid_paper / n_neurons:.2f}%)')

print('\n=== Filtri adattivi (modello sotto-addestrato) ===')
print(f'  THR_SIM_ADAPT  = mean + 3*std = {THR_SIM_ADAPT:.4f}')
print(f'  THR_RATIO_ADAPT = {THR_RATIO_ADAPT}')
print(f'  Neuroni validi (adattivi): {n_valid_adapt}  ({100.0 * n_valid_adapt / n_neurons:.2f}%)')

## 8. Costruzione e salvataggio delle concept map

Salvo **due** file CSV con la stessa struttura (`neuron_id, concept, top1_sim, top2_sim, ratio, valid`):

1. **`concept_map_paper.csv`** — applica i filtri rigorosi dell'Appendice A (`THR_SIM_PAPER=0.42`, `THR_RATIO_PAPER=2.0`). Atteso vuoto (0 validi) sul checkpoint sotto-addestrato corrente. Serve come riferimento riproducibile per quando il modello sara' ri-addestrato su piu' dati.

2. **`concept_map_adaptive.csv`** — applica i filtri adattivi derivati dalla distribuzione osservata (`THR_SIM_ADAPT ≈ mean + 3σ`, `THR_RATIO_ADAPT=1.10`). 

Entrambi contengono *tutti* i `LATENT_DIM` neuroni (6144 o 12288 secondo il modello selezionato); cambia solo il flag `valid`. Cosi' il consumer downstream puo' scegliere il filtro.

In [ ]:
import pandas as pd

terms_arr = np.asarray(terms)
concepts = terms_arr[top1_idx]

def build_concept_map(valid_flags):
    return pd.DataFrame({
        'neuron_id': np.arange(n_neurons, dtype=np.int64),
        'concept': concepts,
        'top1_sim': top1_sim.astype(np.float32),
        'top2_sim': top2_sim.astype(np.float32),
        'ratio': ratio.astype(np.float32),
        'valid': valid_flags.astype(bool),
    })

concept_map_paper = build_concept_map(valid_paper)
concept_map_adapt = build_concept_map(valid_adaptive)

# Nomi con LATENT_DIM: i due modelli (6144 vs 12288) producono concept map distinte.
# Il notebook 11 legge la versione adattiva corrispondente al modello selezionato.
path_paper = os.path.join(OUTPUT_DIR, f'concept_map_paper_{LATENT_DIM}.csv')
path_adapt = os.path.join(OUTPUT_DIR, f'concept_map_adaptive_{LATENT_DIM}.csv')
concept_map_paper.to_csv(path_paper, index=False)
concept_map_adapt.to_csv(path_adapt, index=False)

print(f'concept_map_paper    -> {path_paper}  (validi: {int(concept_map_paper["valid"].sum())})')
print(f'concept_map_adaptive -> {path_adapt}  (validi: {int(concept_map_adapt["valid"].sum())})')

# Per le celle a valle (statistiche, plot) uso la mappa adattiva di default,
# perche' quella paper sara' vuota sul checkpoint corrente.
concept_map = concept_map_adapt

print('\nPrime 10 righe dei neuroni validi (adattivi):')
concept_map[concept_map['valid']].sort_values('top1_sim', ascending=False).head(10)

## 9. Statistiche del naming

Quattro viste:
1. Distribuzione `top1_sim` su tutti i neuroni
2. Distribuzione `top1_sim` sui soli neuroni validi
3. Top 20 concetti piu' frequenti tra i validi
4. Sensibilita' al threshold: numero di validi per soglia in `[0.30, 0.60]` (step 0.05), tenendo fisso `ratio > 2.0`

Tutti i grafici salvati in `CHARTS_DIR`.

In [ ]:
import matplotlib.pyplot as plt

# Per i plot uso 'valid' = mappa adattiva (la paper e' vuota su questo checkpoint).
valid = valid_adaptive
n_valid = int(valid.sum())

# --- 1. Distribuzione top1_sim su tutti i neuroni (entrambe le soglie come vline) ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(top1_sim, bins=60, color='steelblue', edgecolor='black', alpha=0.85)
ax.axvline(THR_SIM_ADAPT, color='green', linestyle='--', label=f'soglia adattiva {THR_SIM_ADAPT:.3f}')
ax.axvline(THR_SIM_PAPER, color='red', linestyle='--', label=f'soglia paper {THR_SIM_PAPER}')
ax.set_xlabel('top1 cosine similarity')
ax.set_ylabel('# neuroni')
ax.set_title('Distribuzione top1_sim — tutti i neuroni')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR, 'plot_top1_sim_all.png'), dpi=120)
plt.show()

# --- 2. Distribuzione top1_sim sui validi (adattivi) ---
fig, ax = plt.subplots(figsize=(8, 4))
if n_valid > 0:
    ax.hist(top1_sim[valid], bins=30, color='seagreen', edgecolor='black', alpha=0.85)
ax.set_xlabel('top1 cosine similarity')
ax.set_ylabel('# neuroni')
ax.set_title(f'Distribuzione top1_sim — validi adattivi (n={n_valid})')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR, 'plot_top1_sim_valid.png'), dpi=120)
plt.show()

# --- 3. Top 20 concetti piu' frequenti tra i validi adattivi ---
fig, ax = plt.subplots(figsize=(9, 6))
if n_valid > 0:
    top20 = concept_map[concept_map['valid']]['concept'].value_counts().head(20)
    ax.barh(top20.index[::-1], top20.values[::-1], color='darkorange', edgecolor='black')
    ax.set_xlabel('# neuroni')
    ax.set_title('Top 20 concetti piu\' frequenti tra i neuroni validi (adattivi)')
else:
    ax.text(0.5, 0.5, 'Nessun neurone valido', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Top 20 concetti — vuoto')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR, 'plot_top20_concepts.png'), dpi=120)
plt.show()

# --- 4. Sensibilita' al threshold (con ratio adattivo + paper) ---
thresholds = np.arange(0.10, 0.45 + 1e-9, 0.025)
counts_adapt = [int(((top1_sim > t) & (ratio > THR_RATIO_ADAPT)).sum()) for t in thresholds]
counts_paper = [int(((top1_sim > t) & (ratio > THR_RATIO_PAPER)).sum()) for t in thresholds]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(thresholds, counts_adapt, marker='o', color='green',  label=f'ratio>{THR_RATIO_ADAPT} (adattivo)')
ax.plot(thresholds, counts_paper, marker='s', color='red',    label=f'ratio>{THR_RATIO_PAPER} (paper)')
ax.axvline(THR_SIM_ADAPT, color='green', linestyle=':',  alpha=0.6, label=f'thr adattivo {THR_SIM_ADAPT:.3f}')
ax.axvline(THR_SIM_PAPER, color='red',   linestyle=':',  alpha=0.6, label=f'thr paper {THR_SIM_PAPER}')
ax.set_xlabel('soglia top1_sim')
ax.set_ylabel('# neuroni validi')
ax.set_title('Sensibilita\' del numero di neuroni validi al threshold')
ax.legend(fontsize=9)
ax.set_yscale('symlog')
fig.tight_layout()
fig.savefig(os.path.join(CHARTS_DIR, 'plot_threshold_sensitivity.png'), dpi=120)
plt.show()

print('\nValidi per soglia (adattivo vs paper):')
print(f'{"thr":>6}  {"adapt(r>1.10)":>14}  {"paper(r>2.0)":>14}')
for t, ca, cp in zip(thresholds, counts_adapt, counts_paper):
    print(f'{t:>6.3f}  {ca:>14d}  {cp:>14d}')